# Value axis with `next`, on Arithmetic

The value-axis model (`core/pynl/value_axis.py`, notes §27) with `plus` used by learned rules and the state also moving
along `next` (`--learned-plus --state-next`). Values are nodes `v0 … vM`; a panel's value is the fact `x(P, A, v)`. The
background facts are `off(U, V, k)` for V = U + k (k = −2…2) and `plus(U, W, V)` for U + W = V. They carry no weights.

For Arithmetic, a learned **state** walks along each row:
* it starts at the first panel's value;
* at each next panel with value W it moves from U to U + W (one learned matrix) or to U − W (another matrix), or along
  `next` by −2…+2 (one matrix per step);
* in a complete row, the state before the last panel is read at the last panel's value (`ck`) and tells the model whether
  the rows add or take away;
* in the question row, the state at the last given panel, at every value V, scores V for the missing panel.

Nothing says "the last value is the sum": the logic decides where a state can go, and the weights learn which state is the
answer (`docs/ARITHMETIC.md`, "Numbers as objects"). Sections: 1 the rules, 2 training on 3×3, 3 one puzzle walked
through, 4 the trained 3×10 model (rows of 10, eight steps), 5 every run.

In [1]:
import os, sys, json, time, warnings
from collections import Counter
warnings.filterwarnings("ignore", message="IProgress not found")
import numpy as np
if os.path.basename(os.getcwd()) == "notebooks": os.chdir("..")   # paths below are relative to the repo root
sys.path[:0] = [os.path.abspath("core/pynl"), os.path.abspath("analysis")]
import neuralogic
neuralogic.set_max_memory_size(32)
from neuralogic.logging import clear_handlers; clear_handlers()
from neuralogic.core import R, Settings
from neuralogic.dataset import Dataset, Sample
from neuralogic.nn import Adam, MSE
import bench_hub_grounding as B          # data loading
import value_axis as VA                  # the template, facts, scoring
from learned_plus_check import reach     # the values a row's state can reach, and by which path

settings = lambda: Settings(optimizer=Adam(lr=1e-3), error_function=MSE(), iso_value_compression=False, chain_pruning=False)
def model_for(M):                        # the run's template: dim 32, kernel and next steps -2..2
    m = VA.build_value_template(32, M, learned_plus=True, state_next=True)
    m.build(settings())
    return m
def sample(pz, y, gid, M):
    return Sample(R.out(gid)[VA.value_target([pz[B.N_CTX + y][0]], M).tolist()], VA.value_facts(pz, gid))
def build(model, data, tag, M):
    return list(model.build_dataset(Dataset([sample(p, y, f"{tag}{i}", M) for i, (p, y, _) in enumerate(data)])))
def form(pz, a):                         # plus: last = sum of the rest; minus: first = sum of the rest
    row = [pz[p][0][a] for p in B.ROWS["r1"]]
    return "plus" if row[-1] == sum(row[:-1]) else "minus"
def report(outs, data, rules, M, label):
    matched, exact, ties, allv, hits = VA.score(outs, data, rules, M)
    arith = {"plus": [0, 0], "minus": [0, 0]}
    for o, (pz, y, _), r in zip(outs, data, rules):
        p = np.asarray(o, dtype=float).reshape(3, M + 1)
        for i, a in enumerate(VA.ATTRS):
            if r[a.capitalize()] == "Arithmetic":
                arith[form(pz, a)][0] += int(p[i].argmax() == pz[B.N_CTX + y][0][a]); arith[form(pz, a)][1] += 1
    print(f"{label}: matched {100 * matched / len(data):.1f}%, all three values right {100 * allv / len(data):.1f}%; exact value "
          + " / ".join(f"{r.split('_')[0]} {100 * hits[r][0] / hits[r][1]:.1f}" for r in VA.RULES)
          + "; Arithmetic " + ", ".join(f"{k} {100 * v[0] / max(1, v[1]):.1f} (of {v[1]})" for k, v in arith.items()))

## 1. The rules

The template on values 0–10. Below are only the rules of the walk (`st`, `ck`) and the two that read them; the
convolution part is the same as in `notebooks/value_axis.ipynb`. `*` marks hidden atoms (they only bind variables),
`{m, n}` a learned matrix.

In [2]:
lines = str(VA.build_value_template(32, 10, learned_plus=True, state_next=True)).splitlines()
facts = Counter(l[:l.find("(")] for l in lines if l.startswith(("val(", "off(", "plus(")))
print("background facts on values 0-10:", ", ".join(f"{k} {n}" for k, n in facts.items()), "\n")
print("\n".join(l for l in lines if ":-" in l and ("st(" in l or "ck(" in l)))

background facts on values 0-10: val 11, off 49, plus 66 

st(P, A, V) :- *col(P, first), {32, 1} x(P, A, V).
st(Q, A, V) :- *next(P, Q), {32, 32} st(P, A, U), *x(Q, A, W), *plus(U, W, V).
st(Q, A, V) :- *next(P, Q), {32, 32} st(P, A, U), *x(Q, A, W), *plus(V, W, U).
st(Q, A, V) :- *next(P, Q), {32, 32} st(P, A, U), *off(U, V, om2).
st(Q, A, V) :- *next(P, Q), {32, 32} st(P, A, U), *off(U, V, om1).
st(Q, A, V) :- *next(P, Q), {32, 32} st(P, A, U), *off(U, V, op0).
st(Q, A, V) :- *next(P, Q), {32, 32} st(P, A, U), *off(U, V, op1).
st(Q, A, V) :- *next(P, Q), {32, 32} st(P, A, U), *off(U, V, op2).
ck(Rw, A) :- *full(Rw), *inrow(L, Rw), *col(L, last), *next(P, L), {32, 32} st(P, A, V), *x(L, A, V).
ctx(A) :- *full(Rw), {32, 32} ck(Rw, A). [aggregation=avg]
o(A, V) :- *qlast(P), {32, 32} h(P, A, V).
o(A, V) :- {32, 32} st(p7, A, V).
dist(type) :- s(type, v0), s(type, v1), s(type, v2), s(type, v3), s(type, v4), s(type, v5), s(type, v6), s(type, v7), s(type, v8), s(type, v9), s(type, v10). [

## 2. Training on 3×3 (values 0–10)

From scratch: 3000 puzzles, 20 epochs, scored on 500 unseen puzzles (6300–6799) every 5 epochs. The full run
(6000 puzzles, job 11962272) gets Arithmetic 99.8.

In [3]:
IRX3 = "data/iravenx/center_single_shuffle_n_3_maxval_10.json"
B.configure(n_cols=3, maxval=10)
train, test = B.load_iravenx(IRX3, 0, 3000), B.load_iravenx(IRX3, 6300, 500)
rules3 = VA.rules_of(IRX3, 6300, 500)
m3 = model_for(10)
t = time.time()
built_train, built_test = build(m3, train, "t", 10), build(m3, test, "u", 10)
print(f"{len(train)} + {len(test)} puzzles grounded in {time.time() - t:.0f} s; "
      f"{sum(int(np.size(v)) for v in m3.state_dict()['weights'].values())} weights")
outs = m3.test(built_test)
report(outs, test, rules3, 10, "untrained")
for ep in range(5, 21, 5):
    t = time.time()
    m3.train(built_train, epochs=5)
    outs = m3.test(built_test)
    report(outs, test, rules3, 10, f"epoch {ep:2d} ({time.time() - t:.0f} s)")

3000 + 500 puzzles grounded in 43 s; 20961 weights


untrained: matched 12.0%, all three values right 0.0%; exact value Constant 0.0 / Progression 26.1 / Distribute 0.0 / Arithmetic 10.5; Arithmetic plus 11.0 (of 181), minus 9.3 (of 75)


epoch  5 (36 s): matched 99.2%, all three values right 98.4%; exact value Constant 100.0 / Progression 99.8 / Distribute 98.4 / Arithmetic 99.6; Arithmetic plus 100.0 (of 181), minus 98.7 (of 75)


epoch 10 (31 s): matched 98.8%, all three values right 94.6%; exact value Constant 100.0 / Progression 94.6 / Distribute 99.2 / Arithmetic 99.2; Arithmetic plus 100.0 (of 181), minus 97.3 (of 75)


epoch 15 (31 s): matched 99.8%, all three values right 99.6%; exact value Constant 100.0 / Progression 100.0 / Distribute 99.7 / Arithmetic 99.6; Arithmetic plus 100.0 (of 181), minus 98.7 (of 75)


epoch 20 (29 s): matched 99.8%, all three values right 99.6%; exact value Constant 100.0 / Progression 100.0 / Distribute 99.7 / Arithmetic 99.6; Arithmetic plus 100.0 (of 181), minus 98.7 (of 75)


## 3. One Arithmetic puzzle, walked through

The state at the question row's last given panel exists only at the values a path of steps reaches; `reach` lists
them without any model. For each, the model's probability that it is the missing value.

In [4]:
QL = f"p{B.ROWS['r3'][-1]}"
def walk(model, pz, y, a, M, tag):
    q = [pz[p][0][a] for p in B.ROWS["r3"]]
    got = reach(q, M, 2)                                     # paths: add, take, mixed, or with a next step
    facts = VA.value_facts(pz, tag)
    qs = [Sample(R.st(QL, a, f"v{v}")[[0.0] * 32], facts) for v in sorted(got)] + [sample(pz, y, tag, M)]
    outs = model.test(list(model.build_dataset(Dataset(qs))))
    p = np.asarray(outs[-1], dtype=float).reshape(3, M + 1)[VA.ATTRS.index(a)]
    return q, got, {v: np.asarray(o, dtype=float) for v, o in zip(sorted(got), outs)}, p

def show(model, data, rules, M, tag, top=None):
    i, a = next((i, a) for i, ((pz, y, _), r) in enumerate(zip(data, rules)) for a in VA.ATTRS
                if r[a.capitalize()] == "Arithmetic" and form(pz, a) == "plus" and len(reach([pz[p][0][a] for p in B.ROWS["r3"]], M, 2)) > 3)
    pz, y, _ = data[i]
    q, got, vecs, p = walk(model, pz, y, a, M, tag)
    ans = pz[B.N_CTX + y][0][a]
    print(f"puzzle {6300 + i}, {a}, plus form; row 1: {[pz[k][0][a] for k in B.ROWS['r1']]}")
    print(f"question row {q} + ? -> answer {ans}; the state exists at {len(got)} values")
    vs = sorted(got, key=lambda v: -p[v])[:top] if top else sorted(got)
    for v in vs:
        print(f"   value {v:3d}  reached by {got[v]:5s}  |state| {np.linalg.norm(vecs[v]):.2f}  P {p[v]:.3f}{'   <- answer' if v == ans else ''}")

show(m3, test, rules3, 10, "w")

puzzle 6300, color, plus form; row 1: [1, 5, 6]
question row [1, 1] + ? -> answer 2; the state exists at 4 values
   value   0  reached by take   |state| 3.60  P 0.000
   value   1  reached by next   |state| 4.26  P 0.000
   value   2  reached by add    |state| 3.89  P 1.000   <- answer
   value   3  reached by next   |state| 2.41  P 0.000


Every value the state reaches has its own vector. Each kind of step multiplies by its own matrix, so states that came by
adding, by taking away or by a `next` step point in different directions; training learned to read which direction
answers (checked on the full runs with `analysis/learned_plus_check.py`). The length of the state is not the signal.

## 4. The trained 3×10 model: rows of 10, eight steps

Checkpoint `experiments/valaxis_n10_m50_learnednext` (job 11962270, 6000 puzzles, epoch 15 picked on selection),
scored here on 200 unseen puzzles. On all 1000 unseen puzzles its Arithmetic exact is 88.7. The state now reaches
about 43 of the 51 values, so the walk alone narrows little: the weights must tell the `plus` path from the `next`
paths.

In [5]:
IRX10, CK10 = "data/iravenx/center_single_shuffle_n_10_maxval_50.json", "experiments/valaxis_n10_m50_learnednext"
B.configure(n_cols=10, maxval=50)
test10, rules10 = B.load_iravenx(IRX10, 6300, 200), VA.rules_of(IRX10, 6300, 200)
m10 = model_for(50)
m10.load(f"{CK10}/template_best.bin")
t = time.time()
outs10 = m10.test(build(m10, test10, "x", 50))
report(outs10, test10, rules10, 50, f"3x10, 200 unseen puzzles ({time.time() - t:.0f} s)")
show(m10, test10, rules10, 50, "y", top=6)

3x10, 200 unseen puzzles (101 s): matched 100.0%, all three values right 96.0%; exact value Constant 100.0 / Progression 100.0 / Distribute 100.0 / Arithmetic 91.7; Arithmetic plus 88.6 (of 44), minus 94.2 (of 52)


puzzle 6301, color, plus form; row 1: [0, 8, 14, 7, 0, 7, 1, 0, 1, 38]
question row [0, 0, 0, 0, 17, 2, 17, 0, 2] + ? -> answer 38; the state exists at 47 values
   value  38  reached by add    |state| 5.13  P 0.918   <- answer
   value  36  reached by next   |state| 4.90  P 0.081
   value  37  reached by next   |state| 5.50  P 0.000
   value  23  reached by next   |state| 5.32  P 0.000
   value  31  reached by next   |state| 5.45  P 0.000
   value  12  reached by next   |state| 5.37  P 0.000


## 5. Every run (1000 unseen puzzles each, from the runs' `best.json`)

The value axis compares only with itself: without `plus` (no arithmetic background), `plus` used by learned rules, the
same with `next` steps, the same with `plus` derived from the zero and `next` (`--peano`), and the rule given (built-in
sums, checks and answers). Arithmetic exact is the argmax of the value distribution.

In [6]:
runs = [("3x3", "valaxis_n3_m10_noplus"), ("3x3", "valaxis_n3_m10_learned"), ("3x3", "valaxis_n3_m10_learnednext"),
        ("3x3", "valaxis_n3_m10_peano"), ("3x3", "valaxis_n3_m10_peanonext"), ("3x3", "valaxis_n3_m10_plus"),
        ("3x10", "valaxis_n10_m50_noplus"), ("3x10", "valaxis_n10_m50_learned"), ("3x10", "valaxis_n10_m50_learnednext"),
        ("3x10", "valaxis_n10_m50_plus")]
print(f"{'data':5s} {'run':30s} {'weights':>8s} {'epoch':>5s} {'matched':>8s} {'Arithmetic':>11s}")
for data_name, run in runs:
    f = f"experiments/{run}/best.json"
    if not os.path.exists(f):
        continue
    b = json.load(open(f))
    a = b["args"]
    kind = ("without plus" if a["no_plus"] else "rule given" if not a.get("learned_plus") else
            "learned" + (", next" if a.get("state_next") else "") + (", zero+next" if a.get("peano") else ""))
    print(f"{data_name:5s} {run.split('_m')[1][3:] + ' (' + kind + ')':30s} {b.get('weights', 0):8d} {b['epoch']:5d} "
          f"{100 * b['unseen']['matched']:8.1f} {100 * b['unseen']['per_rule']['Arithmetic']:11.1f}")

data  run                             weights epoch  matched  Arithmetic
3x3   noplus (without plus)             10689    35     94.4        70.7
3x3   learned (learned)                 15841    10     99.6        98.8
3x3   learnednext (learned, next)       20961    25    100.0        99.8
3x3   peano (learned, zero+next)        15841    15     99.9        99.6
3x3   peanonext (learned, next, zero+next)    20961     5     99.8       100.0
3x3   plus (rule given)                 10817    10     99.8        99.4
3x10  noplus (without plus)             10689    45     84.0        11.1
3x10  learned (learned)                 15841     5    100.0       100.0
3x10  learnednext (learned, next)       20961    15     99.3        88.7
3x10  plus (rule given)                 10817     5    100.0       100.0
